In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

openai_client = OpenAI()

In [2]:
from data import load_documents, create_chunks, build_index

documents = load_documents()
chunks = create_chunks(documents)
index = build_index(chunks)

print("Documents:", len(documents))
print("Chunks:", len(chunks))

Documents: 72
Chunks: 295


In [3]:
from rag import rag

question = "How does the agentic loop keep calling the model until it stops?"

rag_result = rag(
    openai_client,
    index,
    question
)

print(rag_result["answer"])
print("Input tokens:", rag_result["input_tokens"])

It keeps calling the model inside a `while True` loop.

Each turn:

1. The model is called with the current `messages`.
2. If the model returns any `function_call` items, the code runs them, appends the tool outputs, and sets `has_function_calls = True`.
3. If the model returns only a final `message` and no function calls, `has_function_calls` stays `False`.
4. The loop breaks when `has_function_calls == False`.

So the stopping condition is: no function calls in that turn means the agent is done.
Input tokens: 2311


In [4]:
from agent import agent_loop

question = "How does the agentic loop work, and how is it different from plain RAG?"

agent_result = agent_loop(
    openai_client,
    index,
    question
)

print(agent_result["answer"])
print("Search calls:", agent_result["search_calls"])

Iteration #1
Function call: search {"query":"agentic loop plain RAG difference retrieval generation course lesson"}
Function call: search {"query":"agentic loop how it works iterative plan act observe lesson"}
Function call: search {"query":"RAG vs agentic loop lesson retrieval augmented generation agent"}
Iteration #2
The **agentic loop** is a repeated “think → act → observe → think again” cycle.

From the lesson, an agent has three parts:

- **Instructions**: a developer message that defines the role/behavior
- **Tools**: functions the agent can call, like `search`
- **Memory**: the message history, including user messages, model outputs, and tool results

How it works:
1. You send the user question to the LLM.
2. The LLM may return a tool call instead of a final answer.
3. You run the tool and append the result to the conversation.
4. You send the updated history back to the LLM.
5. Repeat until the model gives a final answer with no more tool calls.

The lesson says this is needed 